# Step 2: Text Classification (Zero-Shot)

This notebook assigns each book a **simple category** (Fiction / Nonfiction / Children's Fiction, etc.) using its description.

**Why zero-shot classification?**
- No labeled training data is needed.
- We provide candidate labels (e.g. `"Fiction"`, `"Nonfiction"`) and a pretrained model scores how well each label fits the text.
- Model used: `facebook/bart-large-mnli` (Natural Language Inference — learns whether a hypothesis follows from a premise).

**Pipeline position:**
```
books_cleaned.csv  →  [this notebook]  →  books_with_categories.csv  →  sentiment analysis / dashboard filters
```

## Load cleaned data and map known categories

Raw categories are granular (e.g. *Juvenile Fiction*, *History*). We first map books with known labels to a smaller set of **simple categories**. Books without a mapping will be classified by the model.

In [1]:
# Load cleaned books and map detailed categories to simple groups.
import numpy as np
import pandas as pd

books = pd.read_csv("datasets/books_cleaned.csv")

category_mapping = {
    "Fiction": "Fiction",
    "Juvenile Fiction": "Children's Fiction",
    "Biography & Autobiography": "Nonfiction",
    "History": "Nonfiction",
    "Literary Criticism": "Nonfiction",
    "Philosophy": "Nonfiction",
    "Religion": "Nonfiction",
    "Comics & Graphic Novels": "Fiction",
    "Drama": "Fiction",
    "Juvenile Nonfiction": "Children's Nonfiction",
    "Science": "Nonfiction",
    "Poetry": "Fiction",
}

books["simple_categories"] = books["categories"].map(category_mapping)
print(f"Mapped: {books['simple_categories'].notna().sum():,} | Missing: {books['simple_categories'].isna().sum():,}")

Mapped: 3,743 | Missing: 1,454


## Set up the zero-shot classifier

The Hugging Face `pipeline` wraps the model. For each description it returns a score for every candidate label; we pick the highest-scoring one.

In [2]:
# Initialize zero-shot classification pipeline.
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]

pipe = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,  # -1 = CPU, 0 = GPU
)

def generate_predictions(sequence, categories):
    """Return the label with the highest confidence score."""
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"])
    return predictions["labels"][max_index]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

## Classify books with missing categories

For books where the original category has no mapping, we predict Fiction vs Nonfiction from the description text.

In [ ]:
# Predict categories for books without a mapped label.
from tqdm import tqdm

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

isbns, predicted_cats = [], []
for i in tqdm(range(len(missing_cats))):
    predicted_cats.append(generate_predictions(missing_cats["description"][i], fiction_categories))
    isbns.append(missing_cats["isbn13"][i])

missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_categories": predicted_cats})

## Merge predictions and export

Predicted labels fill in gaps; existing mapped labels are kept unchanged.

In [ ]:
# Merge predictions and save dataset with categories.
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left")
books["simple_categories"] = np.where(
    books["simple_categories"].isna(),
    books["predicted_categories"],
    books["simple_categories"],
)
books = books.drop(columns=["predicted_categories"])

books.to_csv("datasets/books_with_categories.csv", index=False)
print(f"Saved {len(books):,} books to datasets/books_with_categories.csv")